[![Open In Colab](https://google.com)](https://colab.research.google.com/github/hamagami/IS2026/blob/main/Demo_18_Kmeans_interactive.ipynb)

[![Open In Colab](https://google.com)](https://google.comhamagami/IS2026/blob/main/Demo_18_Kmeans_interactive.ipynb)

# k-means: 割り当てとセントロイド更新を操作 / k-means: Interactive assignment and centroid updates

講義対応：k-meansの基本，初期値，尺度，クラスタ形状．CPUで動くインタラクティブ実験です．

Lecture connection: k-means, initialization, feature scales, and cluster shapes. An interactive CPU demonstration.

**Google Colabでの使い方**：この `.ipynb` をアップロードし，「ランタイム → すべてのセルを実行」します．CPUで実行できます．

**Using Google Colab**: Upload this `.ipynb` and select Runtime → Run all. A CPU is sufficient.

タイトル，コメント，説明，表示ラベルは日本語・英語併記です．各ノートブックは独立して動きます．乱数シードを固定しますが，GPUやライブラリの違いで結果は多少変わります．

Titles, comments, explanations, and display labels are provided in Japanese and English. Each notebook runs independently. Seeds are fixed, but results can vary across GPUs and library versions.


## 操作 / How to interact
データ・k・初期値・標準化を選び，Resetを押します．Nextで「割り当て」と「中心更新」を1回ずつ進めます．矢印は更新前の中心から次の平均への移動を表します．データ点は動きません．

Choose the data, k, seed, and standardization, then press Reset. Next alternates between assignment and centroid updates. Arrows show movement from old centers to new means. Data points remain fixed.

$$c_i=\arg\min_j\|x_i-\mu_j\|^2,\quad \mu_j=\frac1{|C_j|}\sum_{i\in C_j}x_i,\quad J=\sum_i\|x_i-\mu_{c_i}\|^2.$$
空クラスタはその中心を維持する方針です．再配置による目的関数のジャンプを避け，単純なLloyd法の下降を観察します．

Empty clusters keep their centers. This avoids objective jumps from relocation and illustrates descent in a simple Lloyd algorithm.


## 準備 / Setup


In [ ]:
import sys,subprocess,importlib.util
for module,package in [('sklearn','scikit-learn'),('ipywidgets','ipywidgets')]:
    if importlib.util.find_spec(module) is None: subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs,make_moons
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from IPython.display import display,clear_output
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError: pass
plt.rcParams.update({'figure.dpi':110,'font.size':11})
COLORS=np.array(['#ff8618','#8b38c5','#00a99d','#2476dc','#d53c64','#96852b'])
def dataset(kind='blobs',seed=7):
    if kind=='moons':return make_moons(n_samples=100,noise=.06,random_state=seed)[0]
    x,_=make_blobs(n_samples=100,centers=[[-2,2],[-2,-2],[2,0]],cluster_std=.65,random_state=seed)
    if kind=='stretched':x=x@np.array([[1.8,.7],[0,.35]])
    if kind=='different scales':x[:,0]*=100
    return x


# 日本語フォントを設定します． / Configure a Japanese font for bilingual figures.
if importlib.util.find_spec('japanize_matplotlib') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'japanize-matplotlib'])
import japanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False


## モデル / Model


In [ ]:
class LloydDemo:
    def reset(self,kind='blobs',k=3,seed=0,standardize=False):
        self.raw=dataset(kind);self.X=StandardScaler().fit_transform(self.raw) if standardize else self.raw.copy()
        self.k=k;self.rng=np.random.default_rng(seed)
        self.centers=self.X[self.rng.choice(len(self.X),k,replace=False)].copy()
        self.labels=None;self.phase='assign';self.iteration=0;self.trace=[];self.movement=None
    def objective(self):
        # 割り当て済みなら，固定したその所属に対するJを計算します． / If assigned, evaluate J using the current fixed memberships.
        if self.labels is None:return np.min(((self.X[:,None]-self.centers)**2).sum(-1),axis=1).sum()
        return ((self.X-self.centers[self.labels])**2).sum()
    def step(self):
        if self.phase=='assign':
            self.labels=((self.X[:,None]-self.centers)**2).sum(-1).argmin(1)
            self.movement=None;self.phase='update';stage='最近傍の中心へ割り当て / Assigned to nearest centers'
        else:
            old=self.centers.copy()
            for j in range(self.k):
                if np.any(self.labels==j):self.centers[j]=self.X[self.labels==j].mean(0)
            self.movement=(old,self.centers.copy());self.iteration+=1;self.phase='assign';stage='中心をクラスタ平均へ更新 / Moved centers to cluster means'
        self.trace.append(float(self.objective()));return stage
    def draw(self,stage='初期中心 / Initial centers'):
        fig,axes=plt.subplots(1,2,figsize=(11,4),gridspec_kw={'width_ratios':[2,1]})
        ax=axes[0];c='slategray' if self.labels is None else COLORS[self.labels]
        ax.scatter(*self.X.T,c=c,s=25)
        ax.scatter(*self.centers.T,c=COLORS[:self.k],s=180,marker='X',edgecolor='black')
        if self.movement is not None:
            for a,b in zip(*self.movement):ax.annotate('',xy=b,xytext=a,arrowprops={'arrowstyle':'->','color':'black'})
        ax.set_title(stage);ax.set_xlabel('特徴量1 / Feature 1');ax.set_ylabel('特徴量2 / Feature 2')
        axes[1].plot(self.trace,'o-');axes[1].set_title('クラスタ内平方和 / Within-cluster SSE');axes[1].set_xlabel('操作回数 / Half-step')
        fig.suptitle(f'反復 / Iteration {self.iteration} | 次の操作 / Next: {dict(assign="割り当て / assign",update="更新 / update")[self.phase]} | J={self.objective():.2f}')
        plt.tight_layout();plt.show()
demo=LloydDemo();demo.reset();demo.draw()


## インタラクティブ操作 / Interactive controls


In [ ]:
kind=widgets.Dropdown(options=[('球状 / Blobs','blobs'),('三日月 / Moons','moons'),('細長い / Stretched','stretched'),('異なる尺度 / Different scales','different scales')],description='データ / Data', style={'description_width':'initial'}, layout=widgets.Layout(width='auto',min_width='260px'))
k=widgets.IntSlider(value=3,min=2,max=6,description='k', style={'description_width':'initial'}, layout=widgets.Layout(width='auto',min_width='260px'))
seed=widgets.IntSlider(value=0,min=0,max=20,description='乱数種 / Seed', style={'description_width':'initial'}, layout=widgets.Layout(width='auto',min_width='260px'))
scale=widgets.Checkbox(value=False,description='標準化 / Standardize', style={'description_width':'initial'}, layout=widgets.Layout(width='auto',min_width='260px'))
reset=widgets.Button(description='初期化 / Reset', style={'description_width':'initial'}, layout=widgets.Layout(width='auto',min_width='260px'),button_style='info');next_button=widgets.Button(description='次の操作 / Next half-step', style={'description_width':'initial'}, layout=widgets.Layout(width='auto',min_width='260px'),button_style='success')
finish=widgets.Button(description='10回反復 / Run 10 iterations', style={'description_width':'initial'}, layout=widgets.Layout(width='auto',min_width='260px'))
out=widgets.Output()
def reset_click(_):
    demo.reset(kind.value,k.value,seed.value,scale.value)
    with out:clear_output(wait=True);demo.draw()
def next_click(_):
    stage=demo.step()
    with out:clear_output(wait=True);demo.draw(stage)
def finish_click(_):
    for _ in range(20):demo.step()
    with out:clear_output(wait=True);demo.draw('割り当てと更新の反復後 / After repeated assignment/update')
reset.on_click(reset_click);next_button.on_click(next_click);finish.on_click(finish_click)
display(widgets.VBox([widgets.HBox([kind,k]),widgets.HBox([seed,scale]),widgets.HBox([reset,next_button,finish]),out]))
reset_click(None)


## 試すこと / Things to try
- 同じデータでkを2，3，4に変える．
  Try k = 2, 3, and 4 on the same data.
- Seedを変え，中心の初期配置による違いを見る．
  Change the seed to observe the effect of centroid initialization.
- different scalesで標準化を切り替える．図も標準化後の座標に変わります．
  Select different scales and toggle standardization. The plot also switches to standardized coordinates.
- moonsで「見た目の曲線」と距離による分割の違いを確認する．
  Select moons and compare the visible curved groups with the distance-based partition.

ウィジェットが表示されない環境でも，`demo.reset('moons',3,0,False)` → `demo.step()` → `demo.draw()` をセルから実行できます．

If widgets are unavailable, run `demo.reset('moons',3,0,False)`, then `demo.step()` and `demo.draw()` directly in a cell.


## 参考 / References
- [scikit-learn k-means](https://scikit-learn.org/stable/modules/clustering.html#k-means)
